<table align="left">

  <h1> Options to run the notebook </h1>
 
  <td>
    <a href="https://colab.research.google.com/github/Dong-Xuyong/Aprendizagem/blob/main/Projeto/ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/Dong-Xuyong/Aprendizagem/blob/main/Projeto/ml.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

# Set up

In [ ]:
import urllib.request
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_curve
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score, recall_score

from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
url = "https://raw.githubusercontent.com/Dong-Xuyong/Aprendizagem/main/Projeto/Dataset/df_final.zip"

urllib.request.urlretrieve(url, "df.zip")

with zipfile.ZipFile("df.zip", 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
df = pd.read_csv("df.csv")

In [ ]:
columns_cat = ["typeClient"]
columns_num = ["Week", "SalesUnitWeek", "SalesPesosWeek", "ReturnsUnitNextWeek", "RetrunsPesosNestWeek",
               "Latitude", "Longitude", "weight", "salesDepot"]

In [ ]:
num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy="median")),
        ('std_scaler', StandardScaler()),
    ])

num_attribs = columns_num
cat_attribs = columns_cat

full_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("cat", OneHotEncoder(), cat_attribs),
    ])

In [ ]:
#Split the target from dataset

X = df.drop('AjustedDemand', axis=1)
X = full_pipeline.fit_transform(X)
y = df['AjustedDemand'].copy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Decision Tree

In [ ]:
# Decision Tree

sgd_clf = SGDClassifier(max_iter=1000, tol=1e-3, random_state=42)
sgd_clf.fit(X_train, y_train)

In [ ]:
y_pred_DT = sgd_clf.predict(X_test)
precision_score(y_test, y_pred_DT)

In [ ]:
confusion_matrix(y_pred_DT, y_test)

In [ ]:
# Random Forest
forest_clf = RandomForestClassifier(n_estimators=100, random_state=42)
forest_clf.fit(X_train, y_train)

In [ ]:
y_pred_RF = sgd_clf.predict(X_test)
precision_score(y_test, y_pred_RF)

In [ ]:
confusion_matrix(y_pred_RF, y_test)

In [ ]:
# XGBoost

from xgboost import XGBClassifier

model = XGBClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
precision_score(y_test, y_pred)

In [ ]:
confusion_matrix(y_pred, y_test)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

clf_gini = DecisionTreeClassifier(criterion='b', max_depth=3, random_state=0)

In [ ]:
# fit the model
clf_gini.fit(X_train, y_train)

In [ ]:
y_pred_gini = clf_gini.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score
print('Model accuracy score with criterion gini index: {0:0.4f}'. format(accuracy_score(y_test, y_pred_gini)))

In [ ]:
y_pred_train_gini = clf_gini.predict(X_train)
y_pred_train_gini

In [ ]:
print('Training-set accuracy score: {0:0.4f}'. format(accuracy_score(y_train, y_pred_train_gini)))

In [ ]:
# print the scores on training and test set

print('Training set score: {:.4f}'.format(clf_gini.score(X_train, y_train)))

print('Test set score: {:.4f}'.format(clf_gini.score(X_test, y_test)))

In [ ]:
df.columns

In [ ]:
plt.figure(figsize=(12,8))

from sklearn import tree

tree.plot_tree(clf_gini.fit(X_train, y_train))

In [ ]:
a = X[23]
clf_gini.predict_proba([a])

In [ ]:
tree = clf_gini.tree_
tree

In [ ]:
tree.node_count

In [ ]:
tree.max_depth

In [ ]:
tree.max_n_classes

In [ ]:
tree.n_features

In [ ]:
tree.n_outputs

In [ ]:
tree.n_leaves

In [ ]:
tree.impurity.round(3)

In [ ]:
tree.children_left[0], tree.children_right[0]

In [ ]:
tree.children_left[3], tree.children_right[3]

In [ ]:
tree.feature

In [ ]:
tree.threshold

In [ ]:
tree.value.round(-3)

In [ ]:
tree.n_node_samples

In [ ]:
np.all(tree.value.sum(axis=(1, 2)) == tree.n_node_samples)

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    'max_leaf_nodes': list(range(2, 100)),
    'max_depth': list(range(1, 7)),
    'min_samples_split': [2, 3, 4]
}
grid_search_cv = GridSearchCV(DecisionTreeClassifier(random_state=42),
                              params,
                              cv=3)

grid_search_cv.fit(X_train, y_train)

In [ ]:
grid_search_cv.best_estimator_

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = grid_search_cv.predict(X_test)
accuracy_score(y_test, y_pred)